# BigAlpha 2026 · 端到端大模型 —— 推理 + 评估

本 notebook 是提交件中**唯一**的 notebook，只负责**推理**：加载 `bigalpha_e2e_train.py` 产出的
`model.json`，在平台注入的测试区间上打分，**不训练**。评估模块调用在最后一节。

训练逻辑（配置 / 模型结构 / 数据构建 / `train_and_save`）全部沉淀在 `bigalpha_e2e_train.py` 里，
作为单一事实来源；此处只 `import` 复用，不重复定义。

## 提交三件套

| 文件 | 作用 |
|---|---|
| `bigalpha_e2e_train.py` | 本地训练脚本 = 单一事实来源。私榜阶段平台调 `train_and_save` 从零重训 |
| `model.json` | 已训练权重（文本 JSON：权重 + normalizer + 结构超参一体） |
| `bigalpha_e2e_predict.ipynb` | 本文件。定义 `main(datasources, start_date, end_date)` + 调用评估模块 |

1. **阶段一（参赛者跑一次）** —— 在三件套所在目录执行 `python bigalpha_e2e_train.py`
   （或把下面「可选：现场训练」的 `RUN_TRAIN` 置 `True`），产出 `model.json`。
2. **阶段二（平台公榜调 `main`）** —— 加载 `model.json`，在注入的测试区间上推理打分。

> 公榜阶段平台只替换 `datasources / start_date / end_date` 并调用 `main`，
> 仅基于提交的权重做推理；私榜阶段平台用 `train_and_save` 在隔离环境从零重训。

## 模型申报

| 项 | 值 |
|---|---|
| 数据源 | `bigalpha_2026_stock_bar5m`（特征）、`bigalpha_2026_instruments`（股票池）、`bigalpha_2026_exposure`（标签 `ret`）|
| 输入字段数 | **25**（≤100）。时序长度不计入字段数 |
| 回看窗口 | **5 个交易日** × 48 根 5 分钟 bar = 240 步（≤240 交易日）|
| 可训练参数量 | 见下方合规自检输出（区间 [1e5, 1e8]）|
| 预训练权重 | **无**，全部参数基于本竞赛数据从零训练 |
| 随机种子 | `SEED = 42`（训练/推理全链路确定性）|
| 预处理 | 仅：缺失值填充；按字段统一 log1p；按字段统一 z-score。**统计量只取训练集** |

## 允许清单之外的操作：无

不做跨字段算子、滚动统计、因子合成、降维、第三方数据。
时序差分以**可训练因果卷积**（`SelfRefStem.diff_k`，nn.Parameter）实现，
跨字段混合由**可学线性层**（`SelfRefStem.contrast`）完成 —— 二者都是网络结构
而非离线特征，网络看不到"哪个通道是 bid_price"这类人工知识。
读出层的 `mean(dim=1)` 作用在 SSM 隐状态上，不是对原始字段做滚动统计。


## 1 导入共享定义与合规自检

In [ ]:
import os
import sys

# 三件套平铺在同一目录：把它加进 sys.path，保证 import 得到训练脚本
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import json
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ==== 从训练侧脚本导入共享定义（配置 / 模型结构 / 数据构建 / 推理辅助）====
# bigalpha_e2e_train.py 是训练与推理的单一事实来源；此处只复用，不重复定义
from bigalpha_e2e_train import (
    FREQ, MODEL_PATH, CACHE_DIR, INFER_CACHE_DIR, INFER_INDEX_DIR,
    submission_config, train_and_save,
    build_infer_cache, make_index_dir, open_cache,
    pick_table, covers, select_dates, predict, load_bundle, _ckpt_n_days,
    build_model, check_fields, check_lookback, check_params,
    check_no_pretrained, check_submission,
    evaluate_scores, format_score_report, BARRA_STYLES,
    available_memory_gb, chunk_size,
)

datasources = {"bar5m": "bigalpha_2026_stock_bar5m"}   # 平台会注入同形状的 dict；命名对齐官方模板

# ---- 合规自检 ----
cfg = submission_config()
check_fields(cfg.data.fields, book_levels=cfg.data.book_levels,
             pre_close=cfg.data.pre_close)
check_lookback(cfg.data.n_days)
_probe = build_model(cfg.data.n_fields, cfg.model,
                    bars_per_day=cfg.data.bars_per_day, n_days=cfg.data.n_days)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print(f"字段数={len(cfg.data.fields)} 回看={cfg.data.n_days}交易日"
      f"×{cfg.data.bars_per_day}bar seed={cfg.train.seed}")
print(f"可训练参数量 = {check_params(_probe):,}")
print("字段清单:", cfg.data.fields)
del _probe

# ---- 规格预检：OOM 要能被预测，而不是撞上才知道 ----
# 内存必须读 cgroup：容器里 /proc/meminfo 报的是宿主机内存（可能上百 GB），
# 照它定 batch 必然 OOM。
_cores = os.cpu_count() or 1
print(f"\n规格: {_cores} 核 / {available_memory_gb():.1f} GB(cgroup 实测) "
      f"/ GPU={torch.cuda.is_available()}")
print(f"取数分批: 每批 {chunk_size(cfg.data.bars_per_day)} 只股票 × 1 个月"
      f"（峰值内存与区间长度、股票总数无关，只由这个批量决定）")
if not torch.cuda.is_available() and _cores <= 2:
    print("→ 本规格只适合**推理**。训练请切 GPU 规格：全量训练集在单核上"
          "一个 epoch 约 2~3 小时，预算护栏会在第 1 个 epoch 后掐掉。")


## 2 可选：现场训练产出权重

正式做法是在终端跑一次 `python bigalpha_e2e_train.py`。权重是**文本 JSON**
（权重 + normalizer + 结构超参一体），转换后做逐位往返自检。

In [ ]:
# 正式做法是在**GPU 规格**上执行 `python bigalpha_e2e_train.py` 训练一次，
# 把产出的 model.json 随三件套上传；本 cell 只是便捷入口。
# 套 `if __name__ == "__main__":`（官方模板同款）：平台以 import 方式提取代码
# 调用 main 时不执行自测，避免每次评测都白跑一遍。
if __name__ == "__main__":
    # 只认显式的 RUN_TRAIN —— 权重缺失时 main 会给明确报错，不会偷偷开训。
    RUN_TRAIN = False

    if RUN_TRAIN:
        # 1C/6GB 这类规格**训不完**：不是内存问题，是时间。全量训练集在单核上
        # 一个 epoch 约 2~3 小时，budget_hours 护栏会在第 1 个 epoch 后收官，
        # 选出来的权重等于没训。要么切 GPU，要么显式用 max_train_days 砍训练集。
        _cores = os.cpu_count() or 1
        if not torch.cuda.is_available() and _cores <= 2 \
                and cfg.train.max_train_days == 0:
            raise RuntimeError(
                f"当前规格 {_cores} 核 / 无 GPU，训不完全量训练集。"
                "请切 GPU 规格后再训；若确实要在本规格上跑，先设 "
                "cfg.train.max_train_days（例如 250）缩小训练集，再重跑本 cell。")
        train_and_save(datasources, cfg=cfg)

    ckpt, embedded_norm = load_bundle(MODEL_PATH)
    check_no_pretrained(ckpt)            # provenance 印章：非本管道产出即拒绝
    assert list(embedded_norm.fields) == cfg.data.fields, \
        "权重内嵌的 normalizer 字段清单与申报配置不符 —— 该权重不属于这套输入，请重训"
    print(f"参数量 {ckpt['n_params']:,} | 选中 epoch {ckpt['best'].get('epoch')} "
          f"| 选型指标 {ckpt['best'].get('select_metric')}")


## 3 平台评测入口 `main`

`main(datasources, start_date, end_date)` —— 第一个参数是 **dict**，不是表名字符串。

**推理索引必须用 `require_label=False`。** 第 2 节那份训练 cache 的索引只收有次日
收益的样本，而评估区间最后一个交易日本来就没有次日收益，用训练索引推理会把整天丢掉，
直接踩中"评估区间内不得缺失任一交易日"。`make_index_dir` 只重算索引（秒级），
6 GB 的 blocks 走软链共享，不用重查一遍 dai。

> 若平台文件系统不支持软链、`make_index_dir` 报 `OSError`：把 `CACHE_DIR` 指向一个
> 不存在的路径，`main` 会按区间当场重查 dai 建推理 cache —— 那正是公榜评测时
> 区间落在预建 cache 之外时实际走的路径，只是要多花一次查询时间。

In [ ]:
def main(datasources, start_date, end_date):
    """平台评测入口 —— 签名与官方模板 `Transformer_modelsave_predict.main` 一致。

    第一个参数是 **dict**（如 `{"bar5m": "bigalpha_2026_stock_bar5m"}`），不是表名。
    公榜阶段平台只替换这三个入参并调用本函数，**仅基于提交的权重做推理，不重训**。

    区间处理：公榜验证集区间不公开，注入的 start/end 可能落在预建 cache 之外，
    此时按注入区间**当场建数据**（起点按交易日历往前推够回看窗口），不依赖预建产物
    —— 与官方模板 `build_dataset` 的做法一致。
    """
    import dai

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"未找到权重 {MODEL_PATH}；请先执行 `python bigalpha_e2e_train.py` "
            "训练并保存，再随 notebook 一并上传")

    cfg = submission_config()
    cfg.data.table = pick_table(datasources, freq=FREQ)
    cfg.data.n_days = _ckpt_n_days(MODEL_PATH, cfg.data.n_days)   # 回看以权重申报的为准
    lo, hi = str(start_date)[:10], str(end_date)[:10]

    if covers(CACHE_DIR, start_date, end_date):
        # 训练 cache 的索引是 require_label=True 建的：区间最后一个交易日没有次日收益，
        # 拿它推理会把整天丢掉 → 踩中"评估区间内不得缺失任一交易日"。
        # 派生一份 require_label=False 的索引（秒级），blocks 大文件走软链共享。
        src = INFER_INDEX_DIR
        man = make_index_dir(CACHE_DIR, src, cfg.data.n_days, require_label=False)
        # instrument_id → '000001.SZ' 的映射不在共享文件里，单独带过去
        shutil.copy2(Path(CACHE_DIR) / "id2code.json", Path(src) / "id2code.json")
    else:
        src = INFER_CACHE_DIR
        man = build_infer_cache(src, cfg, start_date, end_date)
    if man.get("source") not in (None, f"dai:{cfg.data.table}"):
        raise ValueError(f"cache 来源 {man.get('source')} 与注入的 {cfg.data.table} 不符")

    _, _, _, _, meta = open_cache(src)
    idx = select_dates(man, meta, lo, hi)
    df = predict(MODEL_PATH, src, indices=idx, id_col="instrument")

    # 与官方模板一致：date 用 datetime64（instruments 表也是），再 inner join 对齐成分股
    df["date"] = pd.to_datetime(df["date"])
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    stk["instrument"] = stk["instrument"].astype(str)
    out = (pd.merge(df, stk, on=["date", "instrument"], how="inner")
             .replace([np.inf, -np.inf], np.nan).dropna(subset=["score"])
             .drop_duplicates(["date", "instrument"])[["date", "instrument", "score"]]
             .reset_index(drop=True))
    assert len(out), "与 instruments 对齐后为空 —— 检查 date 类型与区间"

    # 交易日完整性：注入区间内的每个交易日都必须有分数（缺一天即判无效）
    missing = sorted({str(d)[:10] for d in stk["date"]}
                     - {str(d)[:10] for d in out["date"]})
    assert not missing, f"缺 {len(missing)} 个交易日, 例: {missing[:5]}"

    # 覆盖度：分母必须是当日成分股名单。out 里没打上分的股票是**整行不存在**，
    # 不是 NaN —— 拿 out 自己算 isna() 恒为 0，40% 这条平台硬校验就永远测不出来。
    rep = check_submission(out.assign(date=out["date"].astype(str).str[:10]),
                           id_col="instrument", universe=stk)
    print(f"覆盖度: 最差单日缺失 {rep['worst_daily_missing']:.2%}"
          f"（{rep['worst_missing_date']}）中位 {rep['median_daily_missing']:.2%}"
          f" / {rep['n_universe_days']} 个交易日")
    return out


## 4 推理 → `date / instrument / score`

第 4–7 节的自测 cells 全部套在 `if __name__ == "__main__":` 下（官方模板
`Transformer_modelsave_predict.py` 的同款做法）：平台以 import 方式提取代码、
注入区间调用 `main` 时不会执行，只有参赛者交互运行 notebook 时才跑。
自测窗口 = 2024 全年（训练集止于 2023-12-31，全年纯样本外）。

In [ ]:
# 平台评测不跑本 cell（__main__ 保护）：平台注入区间并直接调 main。
# 交互自测窗口取 2024 全年 —— 训练集止于 2023-12-31，全年纯样本外；评估器计算
# 多日前瞻收益会截掉尾部约 1 个月，全年窗口才有约 230 个有效 IC 日，指标才有
# 统计意义（官方模板自测同为 2024 全年）。1C/6GB 规格上全年推理很慢，先切 4C/16G+。
# 变量命名对齐官方模板（datasources / start_date / end_date），兼容平台可能的同名替换。
if __name__ == "__main__":
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"

    t0 = time.time()
    scores = main(datasources, start_date, end_date)
    print(f"推理耗时 {(time.time() - t0) / 60:.1f} min")
    print(scores.shape, list(scores.columns))
    print(scores.head())


## 5 提交前校验

In [ ]:
if __name__ == "__main__":
    import dai

    # 对应比赛页「数据校验」三条：列名严格三列无多余列；评估区间交易日不缺失；
    # 每个交易日缺失率 ≤40%。
    # 两个分母都必须来自官方 instruments 表，不能拿 scores 自己的内容当期望值：
    #   - expected_dates 用官方交易日历，否则缺日检查永远不会失败；
    #   - universe 用当日成分股名单，否则缺失率退化成 score.isna()，而没打上分的
    #     股票是整行不存在、不是 NaN —— 恒为 0，40% 这条同样永远不会失败。
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [start_date, end_date]}).df()
    expected = sorted({str(d)[:10] for d in stk["date"]})

    report = check_submission(scores.assign(date=scores["date"].astype(str).str[:10]),
                              expected_dates=expected, id_col="instrument",
                              universe=stk)
    assert report["coverage_checked"], "覆盖度未真正校验 —— universe 没传进去"
    print(f"校验通过: {report}  （期望 {len(expected)} 个交易日）")

    per_day = scores.groupby("date")["instrument"].nunique()
    uni_day = stk.assign(_d=stk["date"].astype(str).str[:10]).groupby("_d")["instrument"].nunique()
    print(f"每日打分股票数 min={per_day.min()} median={int(per_day.median())} max={per_day.max()}"
          f"  |  当日成分股数 min={uni_day.min()} median={int(uni_day.median())}")


## 6 本地四分项自评

In [ ]:
if __name__ == "__main__":
    import dai

    # 复刻官方 Score_final = 0.25×(Rank_IC_mean + Rank_IC_IR + Rank_SR + Rank_Stress)，
    # 含官方 DataProcess 全链：均值±3σ winsorize → 截面 z-score →
    # BARRA 十风格 + 中信一级行业哑变量逐日 OLS 残差化
    # （2026-07-30 平台模块 bigalpha_eval v4 内省确认的口径）。
    # 标签与残差化自变量同在 exposure 表，一次查询取齐。
    # LIMIT 1 也必须带 filters：平台对 exposure 表禁止无分区范围的扫描
    ecols = dai.query("SELECT * FROM bigalpha_2026_exposure LIMIT 1",
                      filters={"date": [start_date, end_date]}).df().columns.tolist()
    _up = {c.upper().replace("_", ""): c for c in ecols}
    styles = [_up[s] for s in BARRA_STYLES if s in _up]
    assert len(styles) == len(BARRA_STYLES), f"exposure 缺风格列，实际: {ecols}"
    inds = [c for c in ecols
            if c not in {"date", "instrument", "ret", "weights", "float_market_cap",
                         *styles} and not c.lower().endswith("_code")]
    xcols = styles + inds
    print(f"残差化自变量: {len(styles)} 风格 + {len(inds)} 行业 = {len(xcols)} 列")

    sel = ", ".join(["date::DATE::DATETIME AS date", "instrument",
                     "m_lead(ret, 1) AS label"] + xcols)
    lab = dai.query(f"SELECT {sel} FROM bigalpha_2026_exposure",
                    filters={"date": [start_date, end_date]}).df()
    lab["date"] = lab["date"].astype(str).str[:10]
    lab["instrument"] = lab["instrument"].astype(str)

    df = scores.assign(date=scores["date"].astype(str).str[:10]).merge(
        lab, on=["date", "instrument"], how="inner")
    print(f"对齐后 {len(df):,} 行 / {df['date'].nunique()} 个交易日")
    print(format_score_report(evaluate_scores(df, xcols=xcols),
                              title="本地四分项自评（官方 DataProcess 全链）"))
    # 对照：不残差化的同一份分数 —— 两者之差就是被十风格+行业吃掉的部分
    print(format_score_report(evaluate_scores(df),
                              title="对照：残差化前（**非**官方口径）"))


## 7 调用评估模块

In [ ]:
# 分数经风格剔除后等价于每日更新的单因子，show=True 会画 IC / 分组 / 压力期绩效图。
# 调用方式取自官方模板 Transformer_modelsave_predict.py 末尾（模板同样置于 __main__ 块内）。
if __name__ == "__main__":
    from bigmodule import M

    result = M.bigalpha_eval._latest(factor_data=scores, show=True)
